# RE:SOURCE AI | TASK 1: Reference Experiment

**Project:** Autonomous recovery and scientific verification of ML/DL experiments  
**Model:** Random Forest · **Dataset:** Iris · **Environment:** Google Colab (CPU)

**Day 1 objective:** Produce an original, working, *documented* experiment. Our future agent will attempt to recover a deliberately broken copy and prove that its repair does not change the research methodology.

**Two gaps we will later evaluate:**
- Running successfully does not necessarily mean the original scientific results were reproduced.
- A code repair must preserve the original dataset, train/test split, preprocessing, architecture/parameters, and evaluation procedure.

No AI repair agent or fault injection is used in this notebook. We first need an honest, reproducible reference.

## 1. Import our tools

`scikit-learn` provides the Iris dataset and Random Forest classifier. `NumPy` handles arrays. `hashlib` creates fingerprints so we can detect unexpected dataset or configuration changes. `joblib` saves our trained model.

Colab normally includes these packages. If an import fails, run `%pip install scikit-learn numpy joblib` in a separate cell and restart the runtime.

In [10]:
import json
import hashlib
import platform
from pathlib import Path

import numpy as np
import sklearn
import joblib
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

print("Python:", platform.python_version())
print("Scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("Joblib:", joblib.__version__)

Python: 3.13.18
Scikit-learn: 1.6.1
NumPy: 2.4.6
Joblib: 1.6.0


## 2. Create the project folders

These folders separate the **reference** (what the original experiment did) from later **reports**. In Colab they live under `/content/RE_SOURCE_AI/`. Colab storage is temporary, so download the output ZIP at the end.

In [11]:
ROOT = Path('/content/RE_SOURCE_AI') if Path('/content').is_dir() else Path.cwd() / 'RE_SOURCE_AI'
REF = ROOT / 'reference'
REPORTS = ROOT / 'reports'
REF.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)


def save_json(path, data):
    path.write_text(json.dumps(data, indent=2, sort_keys=True, ensure_ascii=False) + '\n', encoding='utf-8')


def fingerprint(data):
    return hashlib.sha256(data).hexdigest()


def canonical_json(data):
    return json.dumps(data, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode('utf-8')

print('Working directory:', ROOT)

Working directory: /content/RE_SOURCE_AI


## 3. Lock the scientific methodology

This is our **experiment contract**. We choose a fixed random seed, an 80/20 stratified split, no added preprocessing, and a known Random Forest configuration.

The `methodology_sha256` is a fingerprint of these settings. Later, the recovery agent must compare its proposed changes against this contract. A changed seed, split, or model parameter cannot silently pass as the original experiment.

In [12]:
CONFIG = {
    'experiment_id': 'iris_random_forest_v1',
    'dataset': {
        'name': 'Iris',
        'source': 'sklearn.datasets.load_iris',
        'expected_samples': 150,
        'features': 4,
        'classes': 3,
    },
    'preprocessing': {'steps': [], 'description': 'Raw numeric features; no scaling or imputation'},
    'split': {'test_size': 0.20, 'random_state': 42, 'stratify': True},
    'model': {
        'name': 'RandomForestClassifier',
        'parameters': {
            'n_estimators': 100,
            'max_depth': 4,
            'random_state': 42,
            'n_jobs': 1,
        },
    },
    'evaluation': {'metrics': ['accuracy', 'macro_f1'], 'labels': [0, 1, 2]},
}

METHOD_SHA = fingerprint(canonical_json(CONFIG))
save_json(REF / 'experiment_config.json', CONFIG)
print('Experiment ID:', CONFIG['experiment_id'])
print('Methodology SHA-256:', METHOD_SHA)

Experiment ID: iris_random_forest_v1
Methodology SHA-256: 3d853b0c640c21a485954b317b8f65e3027fd8270eb95f8a3e48bfca8817a4de


## 4. Load Iris and save the exact train/test split

Iris is bundled with scikit-learn, so we do not depend on a changing external download link. We record a dataset fingerprint and save the exact training and testing indices. **Stratification** keeps the three flower classes represented in both sets.

Saving the indices matters: even if a future library version changes a split implementation, we retain the original partition.

In [13]:
iris = load_iris()
X = np.asarray(iris.data, dtype=np.float64)
y = np.asarray(iris.target, dtype=np.int64)
assert X.shape == (150, 4) and y.shape == (150,)

# Stable data fingerprint based on numeric content, array shapes, and feature names.
dataset_bytes = (
    canonical_json({'shape_X': list(X.shape), 'shape_y': list(y.shape),
                    'features': list(iris.feature_names), 'target_names': list(iris.target_names)})
    + np.asarray(X, dtype='<f8').tobytes(order='C')
    + np.asarray(y, dtype='<i8').tobytes(order='C')
)
DATA_SHA = fingerprint(dataset_bytes)
all_indices = np.arange(len(y), dtype=np.int64)
train_idx, test_idx = train_test_split(
    all_indices,
    test_size=CONFIG['split']['test_size'],
    random_state=CONFIG['split']['random_state'],
    stratify=y,
)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

np.savez_compressed(REF / 'split_indices.npz', train_idx=train_idx, test_idx=test_idx)
SPLIT_SHA = fingerprint(
    np.asarray(train_idx, dtype='<i8').tobytes(order='C')
    + np.asarray(test_idx, dtype='<i8').tobytes(order='C')
)
save_json(REF / 'dataset_manifest.json', {
    'dataset_sha256': DATA_SHA,
    'dataset_source': CONFIG['dataset']['source'],
    'n_samples': len(y),
    'n_train': len(train_idx),
    'n_test': len(test_idx),
    'feature_names': list(iris.feature_names),
    'target_names': list(iris.target_names),
    'split_indices_sha256': SPLIT_SHA,
})

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(y))} classes')
print(f'Split: {len(train_idx)} training / {len(test_idx)} testing')
print('Dataset SHA-256:', DATA_SHA)
print('Split SHA-256:', SPLIT_SHA)

Dataset: 150 samples, 4 features, 3 classes
Split: 120 training / 30 testing
Dataset SHA-256: 7d29266b13dbddb9753b1b381a764e7cd57f17115cbfe4139478cb0eb44be03d
Split SHA-256: 9dce6c3b038ddc1b1c2e69c37ab9a0d8969bd209cd333d48fb88f7e3019dfdc1


## 5. Train and evaluate the **original** Random Forest

The model learns from the training set only. We evaluate predictions on the held-out test set.

- **Accuracy:** fraction of correctly classified test examples.
- **Macro-F1:** gives equal weight to each class, so good performance on one class cannot hide poor performance on another.

We'll store these values as the original benchmark. A future repair cannot claim successful reproducibility merely by running without errors.

In [14]:
model = RandomForestClassifier(**CONFIG['model']['parameters'])
model.fit(X_train, y_train)
predictions = model.predict(X_test)

accuracy = float(accuracy_score(y_test, predictions))
macro_f1 = float(f1_score(y_test, predictions, average='macro'))
matrix = confusion_matrix(y_test, predictions, labels=CONFIG['evaluation']['labels']).tolist()

RESULT = {
    'experiment_id': CONFIG['experiment_id'],
    'methodology_sha256': METHOD_SHA,
    'dataset_sha256': DATA_SHA,
    'split_indices_sha256': SPLIT_SHA,
    'accuracy': accuracy,
    'macro_f1': macro_f1,
    'confusion_matrix': matrix,
    'test_samples': int(len(y_test)),
}
ENVIRONMENT = {
    'python_version': platform.python_version(),
    'scikit_learn_version': sklearn.__version__,
    'numpy_version': np.__version__,
    'joblib_version': joblib.__version__,
    'platform': platform.platform(),
}

save_json(REF / 'original_metrics.json', RESULT)
save_json(REF / 'environment.json', ENVIRONMENT)
joblib.dump(model, REF / 'original_model.joblib')  # Do not load joblib files from untrusted sources.

print(f'Original accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)')
print(f'Original macro-F1: {macro_f1:.4f}')
print('Confusion matrix:', matrix)
print('\nClassification report:\n')
print(classification_report(y_test, predictions, target_names=iris.target_names))

Original accuracy: 0.9667 (96.67%)
Original macro-F1: 0.9666
Confusion matrix: [[10, 0, 0], [0, 9, 1], [0, 0, 10]]

Classification report:

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



## 6. Verify repeatability inside this runtime (1st gap foundation)

We train the same model again with the same dataset, exact saved split, and parameters. Its test predictions should match the original predictions here.

**Important:** This is a repeatability check *within the current environment*, not proof that results will be identical on every computer or every future scikit-learn version.

In [15]:
second_model = RandomForestClassifier(**CONFIG['model']['parameters'])
second_model.fit(X_train, y_train)
second_predictions = second_model.predict(X_test)

same_predictions = bool(np.array_equal(predictions, second_predictions))
assert same_predictions, 'Repeatability check failed: predictions changed.'
print('PASS: Two runs produced identical test predictions in this runtime.')
print('Recorded original test accuracy:', RESULT['accuracy'])

PASS: Two runs produced identical test predictions in this runtime.
Recorded original test accuracy: 0.9666666666666667


## 7. Demonstrate our methodology guard (2nd gap foundation)

A future repair agent might secretly change `max_depth` to make code run. We should not accept that as an unchanged experiment. The example below *does not modify the reference files*: it creates an altered copy solely to demonstrate detection.

In [16]:
import copy


def methodology_unchanged(proposed_config):
    return fingerprint(canonical_json(proposed_config)) == METHOD_SHA

original_ok = methodology_unchanged(CONFIG)
altered = copy.deepcopy(CONFIG)
altered['model']['parameters']['max_depth'] = 2
altered_ok = methodology_unchanged(altered)

assert original_ok and not altered_ok
print('Original configuration:', 'PASS' if original_ok else 'FAIL')
print('Altered max_depth:', 'REJECTED (methodology changed)' if not altered_ok else 'MISSED CHANGE')
print('Note: A later version can explicitly allow equivalent repairs with evidence.')

Original configuration: PASS
Altered max_depth: REJECTED (methodology changed)
Note: A later version can explicitly allow equivalent repairs with evidence.


## 8. Write a baseline report and package our artifacts

We now have a reference experiment containing its configuration, dataset fingerprint, exact split, original performance, saved model, and environment. These files become the baseline for our future *broken experiment → repair → independent verification* tests.

In [17]:
import shutil

report = f'''# RE:SOURCE AI – Day 1 reference experiment

- Experiment: {CONFIG['experiment_id']}
- Dataset: Iris (bundled with scikit-learn)
- Model: RandomForestClassifier
- Train/test split: {len(train_idx)}/{len(test_idx)}; stratified; seed 42
- Preprocessing: none
- Accuracy: {accuracy:.4f}
- Macro-F1: {macro_f1:.4f}
- Repeatability in current runtime: {'PASS' if same_predictions else 'FAIL'}
- Methodology change test: {'PASS' if original_ok and not altered_ok else 'FAIL'}
- Dataset fingerprint: {DATA_SHA}
- Split fingerprint: {SPLIT_SHA}
- Methodology fingerprint: {METHOD_SHA}
- scikit-learn version: {sklearn.__version__}

This is a reference run, not an autonomous agent recovery result.
'''
(REPORTS / 'baseline_report.md').write_text(report, encoding='utf-8')

zip_path = ROOT.parent / 'RE-SOURCE AI Output.zip'
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir=ROOT)

print('Saved artifacts:')
for file_path in sorted(ROOT.rglob('*')):
    if file_path.is_file():
        print(' -', file_path.relative_to(ROOT))
print('ZIP bundle:', zip_path)

Saved artifacts:
 - reference/dataset_manifest.json
 - reference/environment.json
 - reference/experiment_config.json
 - reference/original_metrics.json
 - reference/original_model.joblib
 - reference/split_indices.npz
 - reports/baseline_report.md
ZIP bundle: /content/RE-SOURCE AI Output.zip


## 9. Download your work before closing Colab

Colab sessions are temporary. Open the left **Files** sidebar and download `RE-SOURCE AI Output.zip`, or use the optional cell below to start a download. The ZIP contains the reference data and reports (not this notebook). Save this notebook separately with **File → Download → Download .ipynb**, or save a copy to Drive.

**Next task (Day 1, Task 2):** Create a separate deliberately broken experiment. Keep `reference/` untouched so the future agent cannot cheat by editing the ground truth.

In [18]:
# Optional: initiate download of the generated ZIP when running in Colab.
# Leave this cell disabled if you only want to use the Colab Files sidebar.
DOWNLOAD_NOW = False
if DOWNLOAD_NOW:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except ImportError:
        print('Not in Colab. ZIP available at:', zip_path)
else:
    print('To download automatically, set DOWNLOAD_NOW = True and run this cell.')

To download automatically, set DOWNLOAD_NOW = True and run this cell.
